# 02 - Initial Data Preprocessing

**Project:** E-Commerce User Behavior Analysis & Recommendation System  
**Notebook purpose:** Apply processing and cleaning steps that were influenced by findings from initial data exploration. Obtain an intermediate cleaned dataset containing both October and November data.

---

## Environment Setup

This notebook was run on **Kaggle Notebooks** using the dataset attached directly 
from the Kaggle dataset page. No local setup or downloads are required to run it.

### To reproduce this notebook

1. Go to the repository on GitHub:  
   [ecommerce-behavior-analysis](https://github.com/halleepham/ecommerce-behavior-analysis)
2. Download `notebooks/02_initial_data_preprocessing.ipynb`
4. Go to the dataset page on Kaggle:  
   [E-Commerce Behavior Data from Multi-Category Store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store)
5. Click the three dots, then **New Notebook** to open a fresh Kaggle notebook with the dataset already attached
6. Click 'File' -> 'Import Notebook' and select the downloaded file. The dataset will already be attached
7. Run all cells top to bottom

### Data path

All cells in this notebook use the following path to access the raw data:

    /kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store

### Python version and key libraries

| Library | Version |
|---|---|
| Python | 3.12.12 |
| pandas | 2.3.3 |
| numpy | 2.0.2 |
| fastparquet | 2025.12.0 |

* In order to install fastparquet:
    * Under 'Session options' in the notebook tab on the right sidebar, make sure 'INTERNET' is on
    * This setting requires that your Kaggle account is phone verified
    * fastparquet chosen over pyarrow because it allows appending to parquet files

---

## Scope

This notebook reads both complete raw files (`2019-Oct.csv` and `2019-Nov.csv`) in chunks of 500,000 rows at a time to avoid memory issues. Each chunk will be individually cleaned and concatenated into a dataframe depending on if it is October or November data. The two processed dataframes will be saved as a single, combined parquet file for more efficient storage and access.

In [2]:
%pip install fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.0 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import sys
import pandas as pd
import numpy as np
import fastparquet
import gc

# Data paths
data_path = '/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store'
oct_file  = f'{data_path}/2019-Oct.csv'
nov_file  = f'{data_path}/2019-Nov.csv'

# Constants
CHUNK_SIZE = 500_000

# Verify files exist 
print(f'October  file exists : {os.path.exists(oct_file)}')
print(f'November file exists : {os.path.exists(nov_file)}\n')

print(f'Python  : {sys.version}')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')

October  file exists : True
November file exists : True

Python  : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
pandas  : 2.3.3
numpy   : 2.0.2


## Data Processing

**clean_chunk**: Function to clean an individual chunk
* Convert timestamps to datetime
* Fill missing values
* Replace '0' prices with NaN
* Cast values to more efficient datatypes
* Remove duplicates in chunk

In [3]:
def clean_chunk(df):
    # Convert into date time
    df['event_time'] = pd.to_datetime(df['event_time'])

    # Fill NA Columns
    df.fillna({'category_code':'unknown', 'brand':'unknown'}, inplace=True)

    # Replace '0' prices with NaN
    df['price'] = df['price'].replace(0, np.nan)

    # Convert to more efficient datatypes
    df['event_type'] = df['event_type'].astype('category')
    df['brand'] = df['brand'].astype('category')
    
    df['price'] = df['price'].astype('float32')
    df[['product_id', 'category_id', 'user_id']] = df[['product_id', 'category_id', 'user_id']].apply(pd.to_numeric, downcast='integer')

    # Drop duplicates
    df.drop_duplicates(inplace=True, ignore_index=True)
    
    return df

**clean_dataset**: Function to clean entire dataset
* Loads raw dataset in chunks
* Cleans chunks individually using previously defined function
* Combines chunks into a single dataset and returns it

In [4]:
def clean_dataset(input_file):

    chunks = []

    for chunk in pd.read_csv(input_file, chunksize=CHUNK_SIZE):
        clean_chunk(chunk)
        chunks.append(chunk)

    clean_df = pd.concat(chunks, ignore_index=True)

    # Drop duplicates from final df (incase of overlap between chunks)
    clean_df.drop_duplicates(inplace=True, ignore_index=True)

    return clean_df

Export processed dataset:
* Get cleaned oct and nov data
* Export data to parquet for more efficient storage and speed
    * Concatenating the dfs and writing once takes too much memory
    * Append one at a time to limit memory usage
        * Peak memory usage still 26/30 gb
        * Better memory optimization will be used in later notebooks


In [5]:
oct_data_clean = clean_dataset(oct_file)
nov_data_clean = clean_dataset(nov_file)

# Export clean data to parquet file
oct_data_clean.to_parquet('/kaggle/working/ecommerce_oct_nov_clean.parquet', engine='fastparquet', compression='snappy')
nov_data_clean.to_parquet('/kaggle/working/ecommerce_oct_nov_clean.parquet', engine='fastparquet', compression='snappy', append=True)

In [ ]:
del oct_data_clean, nov_data_clean
gc.collect()

* Output saved and uploaded as a private Kaggle dataset at:
    * https://www.kaggle.com/datasets/kylenaluan/ecommerce-data-from-oct-and-nov-cleaned
    * This dataset added in the 'Input' section of the right sidebar
* For access, please contact kylenaluan@gmail.com
* Additionally, the clean dataset will be available in the 'Output' section of your notebook after running the code in the cell above
    * This notebook can be attached in the 'Input' section of the sidebar to access the clean dataset after it is created

Check new dataset info

Observe:
* Data shape
* Data types
* Column value statistics
* Duplicate entries
* Null prices

In [4]:
clean_df = pd.read_parquet('/kaggle/input/datasets/kylenaluan/ecommerce-data-from-oct-and-nov-cleaned/ecommerce_oct_nov_clean.parquet')

In [5]:
clean_df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00+00:00,view,44600062,2103807459595387724,unknown,shiseido,35.790001,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00+00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.200001,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01+00:00,view,17200506,2053013559792632471,furniture.living_room.sofa,unknown,543.099976,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01+00:00,view,1307067,2053013558920217191,computers.notebook,lenovo,251.740005,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04+00:00,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.979980,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [6]:
clean_df.shape

(109820004, 9)

In [7]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109820004 entries, 0 to 109820003
Data columns (total 9 columns):
 #   Column         Dtype              
---  ------         -----              
 0   event_time     datetime64[ns, UTC]
 1   event_type     object             
 2   product_id     int32              
 3   category_id    int64              
 4   category_code  object             
 5   brand          object             
 6   price          float32            
 7   user_id        int32              
 8   user_session   object             
dtypes: datetime64[ns, UTC](1), float32(1), int32(2), int64(1), object(4)
memory usage: 6.1+ GB


In [8]:
clean_df.describe()

,product_id,category_id,price,user_id
count,1.098200e+08,1.098200e+08,1.095633e+08,1.098200e+08
mean,1.176162e+07,2.057710e+18,2.923274e+02,5.366616e+08
std,1.543842e+07,1.949823e+16,3.482940e+02,2.145002e+07
min,1.000365e+06,2.053014e+18,7.700000e-01,1.030022e+07
25%,1.005256e+06,2.053014e+18,6.868000e+01,5.162610e+08
50%,5.100425e+06,2.053014e+18,1.655600e+02,5.326247e+08
75%,1.720053e+07,2.053014e+18,3.603400e+02,5.563223e+08
max,1.000286e+08,2.187708e+18,2.574070e+03,5.799699e+08


In [10]:
print(clean_df.duplicated().sum())

0


In [12]:
clean_df.isna().sum()

event_time            0
event_type            0
product_id            0
category_id           0
category_code         0
brand                 0
price            256657
user_id               0
user_session         12
dtype: int64

In [14]:
clean_df[clean_df['price'].isna()]['event_type'].value_counts()

event_type
view    252199
cart      4458
Name: count, dtype: int64

* 0-priced items only occurred in view/cart events
* There were no purchases/transactions that had items with a price of 0
* This means that there were no 'free' items in the store and 0-priced items from the view/cart events were like due to a data recording error.
* Since purchases aren't affected, any monetary/spending features can be derived without issue.